# Pydantic AI — Typed Research Agent
## Type-First Agent Design with Pydantic AI
⏱ ~12 min

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Esturban/agent/blob/master/examples/64-pydantic-ai/pydantic_ai_workbook.ipynb)

**Pydantic AI** is an alternative agent framework from the Pydantic team — it wraps LLM calls in a fully typed interface with `result_type` enforcement. Instead of parsing free text, you define the output shape as a Pydantic model and the framework handles the rest. This workshop contrasts Pydantic AI's `Agent` with LangGraph's `StateGraph` on the same research task. By the end you will understand *when* each framework fits and *how* `result_type` eliminates output parsing boilerplate.

---

### Workshop Roadmap

| # | Topic |
|---|-------|
| 1 | **Concepts** — Pydantic AI vs LangGraph tradeoffs |
| 2 | **Result Type** — Define `ResearchResult` Pydantic model |
| 3 | **Agent Setup** — `Agent` with `result_type` and system prompt |
| 4 | **Run + Inspect** — `run_sync()` and typed `.data` access |
| 5 | **Framework Contrast** — Same task in LangGraph for comparison |
| ★ | **Exercises + Answer Key** |

### Prerequisites
- Python 3.10+, or Google Colab (install cell below)
- `OPENAI_API_KEY` in `.env` or Colab Secrets

In [ ]:
import sys

def _in_colab():
    try:
        import google.colab; return True
    except ImportError:
        return False

if _in_colab():
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
        "pydantic-ai", "langchain", "langchain-openai", "langgraph", "pydantic", "python-dotenv"])
    print("Colab install complete.")
else:
    print("Local — skipping install.")

In [ ]:
import os
try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
except ImportError:
    from dotenv import load_dotenv; load_dotenv()

key = os.environ.get("OPENAI_API_KEY", "")
print(f"API key ready: {bool(key) and key.startswith('sk-')}")

---
## Part 1 — Pydantic AI vs LangGraph

| | Pydantic AI | LangGraph |
|---|---|---|
| Output typing | `result_type=Model` — always typed | TypedDict state — flexible |
| Graph/flow control | Linear by default | Explicit DAG + conditional edges |
| Best for | Single-call typed extraction | Multi-step stateful workflows |
| Boilerplate | Minimal — `Agent` wraps everything | More explicit — nodes + edges |

**Key insight:** Pydantic AI shines when you want a typed response from one LLM call. LangGraph is better when you need loops, branching, or multi-node state.

In [ ]:
from pydantic import BaseModel, Field

RESEARCH_QUERIES = [
    "The invention of the transformer architecture in machine learning",
    "How gradient descent works in neural networks",
    "The difference between supervised and unsupervised learning",
]

print(f"Test queries: {len(RESEARCH_QUERIES)}")

---
## Part 2 — Define `ResearchResult`

In [ ]:
# result_type=ResearchResult tells the agent to enforce this schema on every response
# Field descriptions become the JSON schema — the LLM uses them as instructions for each field
class ResearchResult(BaseModel):
    topic: str = Field(description="The research topic")
    summary: str = Field(description="A 2-3 sentence factual summary")
    key_facts: list[str] = Field(description="3-5 key bullet-point facts")
    confidence: float = Field(description="Confidence level 0.0-1.0", ge=0.0, le=1.0)

# Preview the schema
import json
print(json.dumps(ResearchResult.model_json_schema(), indent=2))

---
## Part 3 — Agent Setup

In [ ]:
from pydantic_ai import Agent

# 'openai:gpt-4o-mini' — Pydantic AI's model string format: provider:model-name
# result_type= replaces manual JSON parsing — Pydantic validates the response before .data is accessible
agent = Agent(
    "openai:gpt-4o-mini",
    result_type=ResearchResult,
    system_prompt=(
        "You are an expert researcher. Given a topic, produce a factual summary, "
        "3-5 key facts, and a confidence score between 0.0 and 1.0."
    ),
)

print("Agent ready.")
print(f"Model:       openai:gpt-4o-mini")
print(f"Result type: {agent.result_type}")

---
## Part 4 — Run + Inspect Typed Output

In [ ]:
# run_sync() blocks until the agent returns a validated ResearchResult — use agent.run() for async
for query in RESEARCH_QUERIES:
    result = agent.run_sync(query)
    r: ResearchResult = result.data  # fully typed — no parsing needed
    print(f"Topic:      {r.topic}")
    print(f"Confidence: {r.confidence:.2f}")
    print(f"Summary:    {r.summary}")
    print("Key facts:")
    for fact in r.key_facts:
        print(f"  • {fact}")
    print()

---
## Part 5 — Same Task in LangGraph (Framework Contrast)

The equivalent in LangGraph requires more code: a TypedDict state, a node, a graph, and manual `with_structured_output` wiring.

In [ ]:
from typing import TypedDict
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.graph import END, START, StateGraph

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
# with_structured_output() forces the LLM to return a validated Pydantic model via function-calling
researcher = llm.with_structured_output(ResearchResult)

# TypedDict state: explicit keys make node I/O contracts readable — no dict.get() guessing
class ResearchState(TypedDict):
    query: str
    result: dict

def research_node(state):
    r = researcher.invoke([
        SystemMessage(content="You are an expert researcher. Produce a factual summary, key facts, and confidence."),
        HumanMessage(content=state["query"])
    ])
    return {"result": r.model_dump()}

g = StateGraph(ResearchState)
g.add_node("research", research_node)
g.add_edge(START, "research")
g.add_edge("research", END)
# compile() locks the graph topology; without it the graph is just a blueprint, not runnable
lg_app = g.compile()

# Run the first query through LangGraph for comparison
lg_result = lg_app.invoke({"query": RESEARCH_QUERIES[0], "result": {}})
print("LangGraph result (same query):")
print(f"  Topic:      {lg_result['result']['topic']}")
print(f"  Confidence: {lg_result['result']['confidence']:.2f}")
print(f"  Summary:    {lg_result['result']['summary'][:100]}...")
print()
print("Comparison:")
print("  Pydantic AI: 8 lines of setup, agent.run_sync(), result.data")
print("  LangGraph:   20+ lines — state, node, graph, edges, compile, invoke")

---
### Exercise 1 — Add a `sources` Field
Add `sources: list[str]` to `ResearchResult` with description `"2-3 suggested sources or references"`. Re-run the agent. Does it populate the field reliably?

### Exercise 2 — Confidence Threshold Filter
Run all 3 queries and filter out results with `confidence < 0.7`. How many pass? What topics trigger low confidence?

In [ ]:
# Exercise 1 — extended model
class ResearchResultV2(BaseModel):
    topic: str = Field(description="The research topic")
    summary: str = Field(description="A 2-3 sentence factual summary")
    key_facts: list[str] = Field(description="3-5 key bullet-point facts")
    confidence: float = Field(description="Confidence level 0.0-1.0", ge=0.0, le=1.0)
    sources: list[str] = Field(description="2-3 suggested sources or references")

agent_v2 = Agent("openai:gpt-4o-mini", result_type=ResearchResultV2,
                 system_prompt="You are an expert researcher.")
r2 = agent_v2.run_sync(RESEARCH_QUERIES[0]).data
print(f"Sources: {r2.sources}")

# Exercise 2 — confidence filter
CONFIDENCE_THRESHOLD = 0.7
all_results = [agent.run_sync(q).data for q in RESEARCH_QUERIES]
high_conf = [r for r in all_results if r.confidence >= CONFIDENCE_THRESHOLD]
print(f"\nResults above threshold ({CONFIDENCE_THRESHOLD}): {len(high_conf)}/{len(all_results)}")
for r in all_results:
    flag = "✓" if r.confidence >= CONFIDENCE_THRESHOLD else "✗"
    print(f"  {flag} {r.topic[:50]} — confidence={r.confidence:.2f}")

---
*Workshop complete. Next: example 66 — SmolAgents.*